# Modelo para la creacion, mediante PDFs

## Importar librerias

In [1]:
import boto3
import tempfile
import pdfplumber
import os
import subprocess
import whisper
import tempfile
from botocore.exceptions import ClientError
from TTS.api import TTS
from pydub import AudioSegment


C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Primero tomamos un pdf y extraemos el texto

In [2]:
def extraer_texto_pdf_s3(bucket, key):
    s3 = boto3.client("s3")

    with tempfile.TemporaryDirectory() as tmpdir:
        local_pdf = os.path.join(tmpdir, "file.pdf")

        s3.download_file(bucket, key, local_pdf)

        texto = ""

        with pdfplumber.open(local_pdf) as pdf:
            for pagina in pdf.pages:
                contenido = pagina.extract_text()
                if contenido:
                    texto += contenido + "\n"

        return texto

### Separa la letra de la cancion de fondo

In [4]:
def separar_y_subir_audio(archivo, bucket, output_prefix="results"):
    """
    Separa audio con Demucs, convierte WAV a MP3 y sube a S3.
    
    Args:
        archivo (str): ruta del archivo mp3/wav de entrada
        bucket (str): nombre del bucket de S3
        output_prefix (str): carpeta destino dentro de S3
    """
    s3 = boto3.client("s3")

    try:
        nombre = os.path.splitext(os.path.basename(archivo))[0]

        # 1. Ejecutar Demucs
        subprocess.run([
            "demucs",
            "--two-stems=vocals",
            archivo
        ], check=True)

        print("Separación completada")

        output_folder = os.path.join("separated", "htdemucs", nombre)

        # 2. Recorrer resultados
        for root, dirs, files in os.walk(output_folder):
            for file in files:

                if file.endswith(".wav"):
                    wav_path = os.path.join(root, file)
                    mp3_path = wav_path.replace(".wav", ".mp3")

                    # convertir WAV → MP3
                    subprocess.run([
                        "ffmpeg",
                        "-y",
                        "-i", wav_path,
                        "-b:a", "192k",
                        mp3_path
                    ], check=True)

                    # subir a S3
                    s3_key = f"{output_prefix}/{nombre}/{file.replace('.wav', '.mp3')}"
                    s3.upload_file(mp3_path, bucket, s3_key)

                    print(f"Subido: {mp3_path}")

    except Exception as e:
        print("Error:", e)

### Extraer la letra de la cancion

In [6]:
def transcribir_audio_s3(bucket_name, s3_key, model_name="base", language="en"):
    """
    Descarga un audio desde S3, lo transcribe con Whisper y regresa el resultado.

    Args:
        bucket_name (str): nombre del bucket S3
        s3_key (str): ruta del archivo en S3
        model_name (str): modelo de Whisper (tiny, base, small, medium, large)
        language (str): idioma del audio

    Returns:
        dict: {
            "texto_completo": str,
            "segmentos": list
        }
    """

    s3 = boto3.client("s3")

    # Descargar audio desde S3
    obj = s3.get_object(
        Bucket=bucket_name,
        Key=s3_key
    )

    audio_bytes = obj["Body"].read()

    # Guardar en archivo temporal
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        local_file = f.name
        f.write(audio_bytes)

    # Cargar modelo Whisper
    model = whisper.load_model(model_name)

    # Transcribir
    result = model.transcribe(
        local_file,
        language=language
    )

    letra_original = result["text"]
    segmentos = result["segments"]

    # Imprimir resultados (opcional)
    print("LETRA COMPLETA:\n")
    print(letra_original)

    print("\nSEGMENTOS:\n")

    for seg in segmentos:
        print(f"Inicio: {seg['start']:.2f}s")
        print(f"Fin: {seg['end']:.2f}s")
        print(f"Texto: {seg['text']}")
        print("-" * 40)

    return {
        "texto_completo": letra_original,
        "segmentos": segmentos
    }


### Crear la letra con LLM

In [22]:
def llamar_qwen(texto_pdf, letra_original, num_lines):
    client = boto3.client("bedrock-runtime", region_name="us-east-1")

    model_id = "google.gemma-3-12b-it"

    prompt = f"""
SISTEMA: Eres un extractor de datos técnicos. 
Tu objetivo es tomar la INFORMACIÓN del PDF y darle el ritmo de la ESTRUCTURA sugerida.

[DATOS TÉCNICOS DEL PDF]
{texto_pdf}

[ESTRUCTURA RÍTMICA A SEGUIR]
{letra_original}

[REGLAS CRÍTICAS]
- PROHIBIDO: No uses ninguna palabra de la 'ESTRUCTURA RÍTMICA'. 
- OBLIGATORIO: Solo usa conceptos encontrados en el 'DATOS TÉCNICOS DEL PDF'.
- Genera exactamente {num_lines} líneas.
- No saludes, no expliques, solo entrega la letra educativa.

RESULTADO (CANCIÓN TÉCNICA):
"""

    messages = [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ]

    try:
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "maxTokens": 1000,
                "temperature": 0.7,
                "topP": 0.9
            }
        )

        return response['output']['message']['content'][0]['text']

    except ClientError as e:
        return f"Error de cliente: {e}"
    except Exception as e:
        return f"Error inesperado: {e}"


### Clonar voz

In [12]:
def generar_y_mezclar_tts_s3(
    resultado,
    bucket,
    voice_key,
    instrumental_key,
    output_key,
    segmentos_whisper,
    language="en",
    instrumental_gain=-8,
    voice_gain=5
):
    """
    1. Descarga voz de referencia desde S3
    2. Genera TTS temporal por línea
    3. Descarga instrumental
    4. Mezcla voces con instrumental
    5. Exporta mp3 final
    6. Sube resultado a S3
    """

    s3 = boto3.client("s3")

    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

    generated_lines = [
        x.strip()
        for x in resultado.split("\n")
        if x.strip()
    ]

    with tempfile.TemporaryDirectory() as tmpdir:

        # =========================
        # Descargar voz referencia
        # =========================

        local_voice = os.path.join(tmpdir, "voice.mp3")

        s3.download_file(
            bucket,
            voice_key,
            local_voice
        )

        # =========================
        # Descargar instrumental
        # =========================

        local_instrumental = os.path.join(
            tmpdir,
            "instrumental.mp3"
        )

        s3.download_file(
            bucket,
            instrumental_key,
            local_instrumental
        )

        instrumental = AudioSegment.from_mp3(
            local_instrumental
        )

        instrumental = instrumental + instrumental_gain

        final_audio = instrumental

        # =========================
        # Generar y mezclar voces
        # =========================

        max_len = min(
            len(generated_lines),
            len(segmentos_whisper)
        )

        for i in range(max_len):

            try:

                text = generated_lines[i]

                print(f"Generando línea {i}")

                wav_path = os.path.join(
                    tmpdir,
                    f"line_{i}.wav"
                )

                # =========================
                # Generar TTS
                # =========================

                tts.tts_to_file(
                    text=text,
                    speaker_wav=local_voice,
                    language=language,
                    file_path=wav_path
                )

                # =========================
                # Cargar voz generada
                # =========================

                voice = AudioSegment.from_wav(
                    wav_path
                )

                voice = voice + voice_gain

                # =========================
                # Posición del overlay
                # =========================

                start_ms = int(
                    segmentos_whisper[i]["start"] * 1000
                )

                print(
                    f"Overlay línea {i} en {start_ms} ms"
                )

                # =========================
                # Mezclar
                # =========================

                final_audio = final_audio.overlay(
                    voice,
                    position=start_ms
                )

            except Exception as e:

                print(f"Error línea {i}: {e}")

        # =========================
        # Exportar final
        # =========================

        final_local = os.path.join(
            tmpdir,
            "final.mp3"
        )

        final_audio.export(
            final_local,
            format="mp3"
        )

        # =========================
        # Subir a S3
        # =========================

        s3.upload_file(
            final_local,
            bucket,
            output_key
        )

        return f"s3://{bucket}/{output_key}"

In [20]:
def extraer_nombre(path):
    res = ""
    p2 = len(path)-1
    for p1 in range(len(path)):
        
         if "/" not in path[p1:p2]:
            while "." in path[p1:p2]:
                 p2 -= 1
            res += path[p1:p2]
            break
    return res

print(extraer_nombre("inputs/songs/example2.mp3"),)

example2


### Pipline

In [25]:
def pipline(pdf_path,music_path,bucket_name,idioma):

    song_name = extraer_nombre(music_path)
    
    texto = extraer_texto_pdf_s3(bucket_name,pdf_path)
    
    separar_y_subir_audio(music_path,bucket_name)
    
    res = transcribir_audio_s3(bucket_name,f"results/{song_name}/vocals.mp3")
    
    resultado = llamar_qwen(texto,res['texto_completo'],len(res['segmentos']))
    
    ruta = generar_y_mezclar_tts_s3(
    resultado=resultado,
    bucket=bucket_name,
    voice_key=f"results/{song_name}/vocals.mp3",
    instrumental_key=f"results/{song_name}/no_vocals.mp3",
    output_key=f"outputs/{song_name}.mp3",
    segmentos_whisper=res['segmentos'],
    language=idioma
    )

    print(ruta)

pipline("inputs/pdfs/topologia_explicacion.pdf","inputs/songs/example2.mp3","music-project-ia","es")
    

Separación completada
Subido: separated\htdemucs\example2\no_vocals.mp3
Subido: separated\htdemucs\example2\vocals.mp3
LETRA COMPLETA:

 I wish I found some better sounds no one's ever heard. I wish I had a better voice to sing some better words. I wish I found some chords in an order that is new. I wish I didn't have to rhyme every time I sang. I was told when I get older all my fears are trinked, but now I'm insecure and I care what people think. My name's blurry facing back. I care what you think. My name's blurry facing back. I care what you think. I wish I could turn back time to the good old days. When the mama sang us to sleep but now we're stressed out. I wish we could turn back time to the good old days. When the mama sang us to sleep but now we're stressed out. Sometimes a certain smell will take me back to when I was young. How come I'm never able to identify what it's coming from. I'd make a candle out of it if I ever found it, try to sell it. Never sell out of it. I probab